U2 AquaVigia: modelo en streaming, evento de negocio y observabilidad (criterios 4, 5 y 6)

Dimension U2 de Jhenderson Machaca: pH y turbidez.

Este cuaderno cierra el flujo: lee los eventos USGS desde Kafka, mantiene la ventana de las ultimas horas por estacion, aplica el modelo del cuaderno B (pronostico a 1 hora), estima el riesgo, publica un EVENTO DE NEGOCIO (alerta de calidad) en Kafka y envia las metricas al Pushgateway para Grafana.

Requisitos antes de correr: cuaderno B ejecutado (existe la carpeta `artifacts/modelos_jhenderson`), productor USGS o `reproducir` enviando a `agua-usgs-eventos`, y el topic de negocio creado:

`docker exec lambda26-kafka /opt/kafka/bin/kafka-topics.sh --bootstrap-server kafka:9092 --create --topic alerta-eventos --partitions 1 --replication-factor 1`

In [1]:
import os, json, time, math, shutil, urllib.request, urllib.error
import pandas as pd
from datetime import datetime, timezone
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StringType, DoubleType
from pyspark.ml import PipelineModel

INTEGRANTE = "Jhenderson Machaca"
ID = "jhenderson"
VARIABLES = ["ph","turbidez_fnu"]
TOPIC_ENTRADA = "agua-usgs-eventos"
TOPIC_NEGOCIO = "alerta-eventos"
BROKER = "kafka:9092"
PUSHGATEWAY = "http://pushgateway:9091"
BASE = "/opt/s12-u2-usgs"
ARTIFACTS = f"{BASE}/artifacts"
RUTA_MODELOS = f"{ARTIFACTS}/modelos_{ID}"
MODO = "reproduccion"    # "vivo": solo lo nuevo (latest). "reproduccion": desde el inicio del topic (earliest)
DURACION_MIN = 5         # cuanto tiempo corre la demostracion
RIESGO_ALERTA = 0.5      # probabilidad desde la cual se emite el evento de negocio
ENFRIAMIENTO_MIN = 60    # no repetir el evento para la misma estacion y variable antes de este tiempo (tiempo del evento)
LIMPIAR_CHECKPOINT = True

# Direccion que preocupa por variable (criterio documentado, ver tabla en la memoria)
DIRECCION = {"temperatura_c": "alta", "oxigeno_mgl": "baja", "ph": "ambas", "turbidez_fnu": "alta"}

spark = (SparkSession.builder.appName("u2-usgs-negocio").master("local[*]")
         .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
         .config("spark.sql.shuffle.partitions", "1").config("spark.sql.session.timeZone", "America/Lima")
         .config("spark.ui.port", "4041").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

META = json.load(open(f"{RUTA_MODELOS}/meta.json", encoding="utf-8"))
MODELOS = {v: PipelineModel.load(f"{RUTA_MODELOS}/modelo_{v}_1h") for v in VARIABLES if v in META["variables"]}
print("Modelos cargados:", list(MODELOS))
for v in MODELOS:
    m = META["variables"][v]
    print(f"  {v}: {m['ganador']}  sigma(RMSE)={m['sigma']:.4f}  mejora vs persistencia={m['mejora_vs_persistencia_pct']:+.1f} %")

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c96a7d61-25e5-43a5-b5f2-1a3599ebc811;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central
	found org.apache.kafka#kafka-clients;3.9.2 in central
	found at.yawk.lz4#lz4-java;1.11.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.5.0 in central
	found org.apache.hadoop#hadoop-client-api;3.5.0 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.mod

Modelos cargados: ['ph', 'turbidez_fnu']
  ph: LinearRegression  sigma(RMSE)=0.0683  mejora vs persistencia=+9.4 %
  turbidez_fnu: LinearRegression  sigma(RMSE)=8.2011  mejora vs persistencia=+4.2 %


## 1. Verificar la entrada

Que el topic de entrada tiene eventos de mis variables y cuantos son vivos o de reproduccion.

In [2]:
esq = (StructType().add("evento_id", StringType()).add("estacion_id", StringType()).add("variable", StringType())
       .add("codigo_parametro", StringType()).add("valor", DoubleType()).add("fecha_medicion_utc", StringType())
       .add("fecha_ingesta_utc", StringType()).add("retraso_min", DoubleType()).add("origen", StringType())
       .add("fuente", StringType()))
ent = (spark.read.format("kafka").option("kafka.bootstrap.servers", BROKER)
       .option("subscribe", TOPIC_ENTRADA).option("startingOffsets", "earliest").load()
       .select(F.from_json(F.col("value").cast("string"), esq).alias("e")).select("e.*"))
ent.where(F.col("variable").isin(VARIABLES)).groupBy("origen", "variable").count().orderBy("origen", "variable").show()

[Stage 14:>                                                         (0 + 1) / 1]

+------+------------+-----+
|origen|    variable|count|
+------+------------+-----+
|  vivo|          ph| 1993|
|  vivo|turbidez_fnu| 1832|
+------+------------+-----+



## 2. Estado por estacion, pronostico y riesgo

El cuaderno guarda en memoria las ultimas lecturas de 5 minutos por estacion (solo 90 min). Cuando una estacion tiene el bloque mas reciente completo con todas las variables y los rezagos de 5, 10, 15, 30 y 60 min, arma las variables del modelo y calcula:
- pronostico a 1 hora,
- riesgo = probabilidad de salir del rango de advertencia de la estacion (percentil 5 y 95 del entrenamiento), suponiendo error normal con la desviacion del RMSE de validacion.

In [3]:
LAGS = [1, 2, 3, 6, 12]
BUF = {}            # estacion -> {ts5: {variable: valor}}
ULTIMO_PRED = {}    # estacion -> ts5 ya pronosticado
ESTADO = {}         # (estacion, variable) -> dict con ultimo valor, pronostico, riesgo
ULT_ALERTA = {}     # (estacion, variable) -> ts del ultimo evento de negocio
TOT = {"eventos": 0, "lotes": 0, "alertas": 0, "pronosticos": 0}

def ncdf(z): return 0.5 * (1 + math.erf(z / math.sqrt(2)))

def riesgo(est, v, pred):
    u = META["umbrales"].get(f"{est}|{v}")
    s = META["variables"][v]["sigma"]
    if not u or s <= 0: return None, None
    d = DIRECCION.get(v, "ambas")
    p_alta = 1 - ncdf((u["p95"] - pred) / s)
    p_baja = ncdf((u["p05"] - pred) / s)
    p = p_alta if d == "alta" else p_baja if d == "baja" else min(1.0, p_alta + p_baja)
    return p, u

def actualizar_buffer(pdf):
    for r in pdf.itertuples():
        t = int(pd.Timestamp(r.fecha_medicion_utc).timestamp()) // 300 * 300
        BUF.setdefault(r.estacion_id, {}).setdefault(t, {})[r.variable] = r.valor
    for est, d in BUF.items():
        if d:
            tmax = max(d)
            for t in [t for t in d if t < tmax - 5400]: del d[t]

def filas_modelo():
    filas = []
    for est, d in BUF.items():
        completos = [t for t, x in d.items() if all(v in x for v in VARIABLES)]
        if not completos: continue
        T = max(completos)
        if ULTIMO_PRED.get(est, 0) >= T: continue
        for v in MODELOS:
            if not all((T - k * 300) in d and v in d[T - k * 300] for k in LAGS): continue
            fila = {"estacion_id": est, "variable": v, "ts5": T, v: d[T][v]}
            for k in LAGS: fila[f"{v}_l{k}"] = d[T - k * 300][v]
            fila["delta_1h"] = fila[v] - fila[f"{v}_l12"]
            for u in VARIABLES:
                if u != v: fila[u] = d[T][u]
            hora = (T % 86400) / 3600.0
            fila["hora_sin"] = math.sin(hora * 2 * math.pi / 24); fila["hora_cos"] = math.cos(hora * 2 * math.pi / 24)
            filas.append(fila)
        ULTIMO_PRED[est] = T
    return filas

def pronosticar(filas):
    res = []
    if not filas: return res
    df = pd.DataFrame(filas)
    for v in MODELOS:
        sub = df[df["variable"] == v]
        if sub.empty: continue
        cols = ["estacion_id", "ts5"] + META["variables"][v]["features"]
        p = MODELOS[v].transform(spark.createDataFrame(sub[cols])).select("estacion_id", "ts5", v, "prediction").toPandas()
        for r in p.itertuples(index=False):
            res.append({"estacion": r.estacion_id, "variable": v, "ts5": int(r.ts5), "ultimo": float(getattr(r, v)), "pred": float(r.prediction)})
    return res

## 3. Evento de negocio

Cuando el riesgo de salir del rango supera el umbral, se emite una ALERTA DE CALIDAD (estado `emitida`). Despues puede quedar `reconocida` o `descartada` (falsa alarma); esa tasa de falsas alarmas es la medida de negocio que se reporta.

El evento se escribe en el topic `alerta-eventos` con la clave `estacion|variable`. Lleva el pronostico, el riesgo y el motivo para que quien lo reciba pueda decidir sin volver a calcular.

In [4]:
def publicar_negocio(ev):
    fila = [(f"{ev['estacion_id']}|{ev['variable']}", json.dumps(ev, ensure_ascii=False))]
    (spark.createDataFrame(fila, ["key", "value"]).write.format("kafka")
     .option("kafka.bootstrap.servers", BROKER).option("topic", TOPIC_NEGOCIO).save())

def evento_negocio(r, p, u):
    ahora = datetime.now(timezone.utc).isoformat()
    ev = {"tipo": "alerta_calidad", "integrante": INTEGRANTE, "estacion_id": r["estacion"], "variable": r["variable"],
          "valor_actual": round(r["ultimo"], 3), "pronostico_1h": round(r["pred"], 3), "riesgo": round(p, 3),
          "umbral_p05": u["p05"], "umbral_p95": u["p95"], "direccion": DIRECCION.get(r["variable"]),
          "ts_medicion_utc": datetime.fromtimestamp(r["ts5"], timezone.utc).isoformat(), "creado_utc": ahora,
          "estado": "emitida", "fuente": "USGS", "modelo": META["variables"][r["variable"]]["ganador"]}
    return ev

## 4. Metricas hacia Pushgateway

Cada lote reenvia el estado completo (el Pushgateway reemplaza el grupo en cada PUT). Metricas: eventos y alertas acumulados, retraso medio de la publicacion del USGS, duracion del lote y, por estacion y variable, ultimo valor, pronostico y riesgo.

In [6]:
def lbl(**k): return ",".join(f'{a}="{b}"' for a, b in k.items())

def push(retraso, dur_lote):
    L = []
    def g(nombre, ayuda, tipo="gauge"):
        L.append(f"# HELP {nombre} {ayuda}"); L.append(f"# TYPE {nombre} {tipo}")
    g("usgs_eventos_total", "Eventos USGS procesados", "counter"); L.append(f'usgs_eventos_total{{integrante="{ID}"}} {TOT["eventos"]}')
    g("usgs_lotes_total", "Lotes de streaming procesados", "counter"); L.append(f'usgs_lotes_total{{integrante="{ID}"}} {TOT["lotes"]}')
    g("usgs_alertas_total", "Eventos de negocio emitidos", "counter"); L.append(f'usgs_alertas_total{{integrante="{ID}"}} {TOT["alertas"]}')
    g("usgs_retraso_min", "Retraso medio entre medicion y publicacion (min)"); L.append(f'usgs_retraso_min{{integrante="{ID}"}} {retraso:.2f}')
    g("usgs_duracion_lote_s", "Duracion del lote (s)"); L.append(f'usgs_duracion_lote_s{{integrante="{ID}"}} {dur_lote:.3f}')
    g("usgs_ultimo_valor", "Ultima lectura"); g("usgs_pronostico_1h", "Pronostico a 1 hora"); g("usgs_riesgo", "Riesgo de salir del rango")
    for (est, v), s in ESTADO.items():
        l = lbl(integrante=ID, estacion=est, variable=v)
        L.append(f"usgs_ultimo_valor{{{l}}} {s['ultimo']}"); L.append(f"usgs_pronostico_1h{{{l}}} {s['pred']}")
        if s["riesgo"] is not None: L.append(f"usgs_riesgo{{{l}}} {s['riesgo']}")
    cuerpo = ("\n".join(L) + "\n").encode()
    req = urllib.request.Request(f"{PUSHGATEWAY}/metrics/job/u2_usgs/instance/{ID}", data=cuerpo, method="PUT")
    try:
        urllib.request.urlopen(req, timeout=5).read()
    except Exception as e:
        print("  Pushgateway no responde:", e)

## 5. Streaming: Kafka a pronostico a evento de negocio y metricas

`foreachBatch` recibe cada lote en orden de tiempo por estacion, actualiza el estado, pronostica, emite el evento si el riesgo supera el umbral (con enfriamiento) y empuja las metricas. El checkpoint permite reanudar sin repetir lotes.

In [12]:
def lote(df, bid):
    t0 = time.monotonic()
    pdf = df.toPandas()
    pdf = pdf[pdf["variable"].isin(VARIABLES) & pdf["origen"].isin(["vivo", "reproduccion", "replay"])].dropna(subset=["valor"])
    if pdf.empty: return
    pdf = pdf.sort_values("fecha_medicion_utc")
    actualizar_buffer(pdf)
    vivos = pdf[pdf["origen"] == "vivo"]["retraso_min"]
    retraso = float(vivos.mean()) if len(vivos) else 0.0
    TOT["eventos"] += len(pdf); TOT["lotes"] += 1
    nuevas = 0
    for r in pronosticar(filas_modelo()):
        p, u = riesgo(r["estacion"], r["variable"], r["pred"])
        ESTADO[(r["estacion"], r["variable"])] = {"ultimo": r["ultimo"], "pred": r["pred"], "riesgo": p}
        TOT["pronosticos"] += 1
        if p is not None and p >= RIESGO_ALERTA:
            ult = ULT_ALERTA.get((r["estacion"], r["variable"]), 0)
            if r["ts5"] - ult >= ENFRIAMIENTO_MIN * 60:
                publicar_negocio(evento_negocio(r, p, u)); ULT_ALERTA[(r["estacion"], r["variable"])] = r["ts5"]
                TOT["alertas"] += 1; nuevas += 1
    dur = time.monotonic() - t0
    push(retraso, dur)
    print(f"lote {bid}: {len(pdf)} eventos | pronosticos acumulados={TOT['pronosticos']} | alertas nuevas={nuevas} (total {TOT['alertas']}) | {dur:.2f}s")

# Reinicia el estado en memoria (por si se vuelve a correr esta celda con otro umbral)
BUF.clear(); ULTIMO_PRED.clear(); ESTADO.clear(); ULT_ALERTA.clear()
TOT.update({"eventos": 0, "lotes": 0, "alertas": 0, "pronosticos": 0})

CHK = f"{ARTIFACTS}/chk_negocio_{ID}"
if LIMPIAR_CHECKPOINT and os.path.exists(CHK): shutil.rmtree(CHK)

lectura = (spark.readStream.format("kafka").option("kafka.bootstrap.servers", BROKER).option("subscribe", TOPIC_ENTRADA)
           .option("startingOffsets", "latest" if MODO == "vivo" else "earliest").option("maxOffsetsPerTrigger", 3000).load()
           .select(F.from_json(F.col("value").cast("string"), esq).alias("e")).select("e.*"))
q = (lectura.writeStream.foreachBatch(lote).option("checkpointLocation", CHK).trigger(processingTime="20 seconds").start())
print(f"Corriendo {DURACION_MIN} min en modo {MODO}. Abre Grafana (43000) mientras tanto.")
q.awaitTermination(DURACION_MIN * 60)
q.stop()
print("Resumen:", TOT)

Corriendo 5 min en modo reproduccion. Abre Grafana (43000) mientras tanto.


lote 0: 1093 eventos | pronosticos acumulados=16 | alertas nuevas=1 (total 1) | 2.52s
lote 1: 1108 eventos | pronosticos acumulados=38 | alertas nuevas=2 (total 3) | 0.76s
lote 2: 1086 eventos | pronosticos acumulados=60 | alertas nuevas=2 (total 5) | 0.83s


lote 3: 622 eventos | pronosticos acumulados=82 | alertas nuevas=2 (total 7) | 1.46s
Resumen: {'eventos': 3909, 'lotes': 4, 'alertas': 7, 'pronosticos': 82}


In [13]:
# Riesgo maximo por serie: sirve para decidir con datos si el umbral de alerta se alcanza o no.
rs = pd.DataFrame([{"estacion": e, "variable": v, "ultimo": s["ultimo"], "pronostico_1h": round(s["pred"], 3),
                    "riesgo": None if s["riesgo"] is None else round(s["riesgo"], 4)} for (e, v), s in ESTADO.items()])
rs = rs.sort_values("riesgo", ascending=False)
print(rs.head(10).to_string(index=False))
print("riesgo maximo:", rs["riesgo"].max(), "  umbral de alerta:", RIESGO_ALERTA)


     estacion     variable  ultimo  pronostico_1h  riesgo
USGS-02074556           ph     7.0          7.030  0.8464
USGS-01594441           ph     7.7          7.715  0.5874
USGS-01491000 turbidez_fnu     3.6          3.789  0.3432
USGS-02054550           ph     8.4          8.450  0.2336
USGS-01650800 turbidez_fnu     0.4          0.976  0.2203
USGS-01649190 turbidez_fnu     0.4          0.983  0.1894
USGS-02074556 turbidez_fnu     2.3          2.697  0.1866
USGS-01646500 turbidez_fnu     2.7          2.860  0.1665
USGS-01650800           ph     7.8          7.819  0.1181
USGS-01646500           ph     8.7          8.711  0.0974
riesgo maximo: 0.8464   umbral de alerta: 0.5


## 6. Evidencia del evento de negocio

Lectura del topic `alerta-eventos`: lo que recibio el consumidor.

In [14]:
neg = (spark.read.format("kafka").option("kafka.bootstrap.servers", BROKER).option("subscribe", TOPIC_NEGOCIO)
       .option("startingOffsets", "earliest").load()
       .select(F.col("key").cast("string").alias("clave"), F.col("value").cast("string").alias("evento"), "partition", "offset"))
print("eventos de negocio en el topic:", neg.count())
neg.orderBy(F.col("offset").desc()).show(8, truncate=120)

eventos de negocio en el topic: 7
+----------------+------------------------------------------------------------------------------------------------------------------------+---------+------+
|           clave|                                                                                                                  evento|partition|offset|
+----------------+------------------------------------------------------------------------------------------------------------------------+---------+------+
|USGS-02074556|ph|{"tipo": "alerta_calidad", "integrante": "Jhenderson Machaca", "estacion_id": "USGS-02074556", "variable": "ph", "val...|        0|     6|
|USGS-01594441|ph|{"tipo": "alerta_calidad", "integrante": "Jhenderson Machaca", "estacion_id": "USGS-01594441", "variable": "ph", "val...|        0|     5|
|USGS-02074556|ph|{"tipo": "alerta_calidad", "integrante": "Jhenderson Machaca", "estacion_id": "USGS-02074556", "variable": "ph", "val...|        0|     4|
|USGS-01646500|ph|{"tipo

## 7. Volumen real y costo

El volumen se mide del topic. Los precios NO estan escritos: se completan SOLO con los valores vigentes de la pagina oficial del proveedor elegido (con enlace y fecha de consulta) en la memoria.

In [15]:
vol = (spark.read.format("kafka").option("kafka.bootstrap.servers", BROKER).option("subscribe", TOPIC_ENTRADA)
       .option("startingOffsets", "earliest").load()
       .select(F.length("value").alias("bytes"), F.col("timestamp").alias("t")))
r = vol.agg(F.count("*").alias("n"), F.sum("bytes").alias("bytes"), F.min("t").alias("t0"), F.max("t").alias("t1")).first()
dias = max((r["t1"] - r["t0"]).total_seconds() / 86400, 1e-6)
print(f"Eventos en el topic: {r['n']}   bytes: {r['bytes']}   ventana: {dias:.3f} dias")
print(f"Ritmo: {r['n']/dias:,.0f} eventos/dia   {r['bytes']/dias/1e6:,.2f} MB/dia   {r['bytes']/dias*30/1e9:,.3f} GB/mes (extrapolado)")
n_series = (spark.read.format("kafka").option("kafka.bootstrap.servers", BROKER).option("subscribe", TOPIC_ENTRADA)
            .option("startingOffsets", "earliest").load()
            .select(F.from_json(F.col("value").cast("string"), esq).alias("e")).select("e.*")
            .select("estacion_id", "variable").distinct().count())
bytes_ev = r["bytes"] / r["n"]
ev_dia_estable = n_series * 288          # una lectura cada 5 min por serie (estacion, variable)
print(f"Series (estacion x variable): {n_series}   bytes por evento: {bytes_ev:.0f}")
print(f"Regimen ESTABLE: {ev_dia_estable:,} eventos/dia   {ev_dia_estable*bytes_ev/1e6:,.2f} MB/dia   {ev_dia_estable*bytes_ev*30/1e9:,.3f} GB/mes")
print("(la medicion de arriba incluye la carga inicial de 6 h y las lecturas reenviadas para pruebas, asi que sobreestima)")
PRECIO_GB_MES_ALMACENAMIENTO = None   # completar con valor oficial vigente
PRECIO_POR_MILLON_EVENTOS = None      # completar con valor oficial vigente
if PRECIO_GB_MES_ALMACENAMIENTO is None or PRECIO_POR_MILLON_EVENTOS is None:
    print("Precios pendientes: completar con la pagina oficial del proveedor (enlace y fecha).")
else:
    costo = (r['bytes']/dias*30/1e9) * PRECIO_GB_MES_ALMACENAMIENTO + (r['n']/dias*30/1e6) * PRECIO_POR_MILLON_EVENTOS
    print(f"Costo mensual estimado: {costo:,.2f}")

Eventos en el topic: 10692   bytes: 3438311   ventana: 0.349 dias
Ritmo: 30,653 eventos/dia   9.86 MB/dia   0.296 GB/mes (extrapolado)


[Stage 47:>                                                         (0 + 1) / 1]

Series (estacion x variable): 66   bytes por evento: 322
Regimen ESTABLE: 19,008 eventos/dia   6.11 MB/dia   0.183 GB/mes
(la medicion de arriba incluye la carga inicial de 6 h y las lecturas reenviadas para pruebas, asi que sobreestima)
Precios pendientes: completar con la pagina oficial del proveedor (enlace y fecha).
